In [1]:
import pandas as pd
from tqdm import tqdm
import json,re
import os
import matplotlib.pyplot as plt
import seaborn as sns
import shutil
from PIL import Image
import math
import ast

def read_jsonl(file_path):
    data = []
    with open(file_path, "r", encoding="utf-8") as file:
        for idx,line in enumerate(file):
            try:
                json_object = json.loads(line.strip())
                data.append(json_object)
            except json.JSONDecodeError as e:
                print(f"idx {idx}, line {line}")
                print(f"Error decoding JSON: {e}")
                continue
                # 如果选择抛出异常，使用下面这行
                # raise
    return data

def convert_action(action, img_width=720, img_height=1280):
    if action.startswith('Click'):
        # Extract coordinates from 'Click (x,y)' using regex
        coords = re.findall(r'\(([^)]+)', action)
        if coords:
            x, y = coords[0].split(', ')
            json_answer = {
                "action": "click",
                "coordinate": [int(float(x)*img_width),int(float(y)*img_height)]
            }
            return json_answer
            # return f"click(start_box='<|box_start|>({x},{y})<|box_end|>')"
    
    elif action == 'KEY_BACK':
        json_answer = {
            "action": "system_button",
            "button": "Back"
        }
        # return "press_back()"
        return json_answer
    
    elif action == 'KEY_HOME':
        json_answer = {
            "action": "system_button",
            "button": "Home"
        }
        return json_answer
        
    
    elif action.startswith('Stop'):
        json_answer = {
            "action": "terminate",
            "status": "success"
        }
        return json_answer

    elif action.startswith('Type'):
        # Extract text from 'Type: content'
        content = action.split(': ', 1)[-1]
        json_answer = {
            "action": "type",
            "text": content
        }
        return json_answer
        # return f"type(content='{content}')"
    
    elif action.startswith('Swipe'):
        # Extract start and end coordinates from 'Swipe (x1,y1), (x2,y2)'
        coords = re.findall(r'\(([^)]+)', action)
        if len(coords) == 2:
            x1, y1 = coords[0].split(', ')
            x2, y2 = coords[1].split(', ')
            json_answer = {
                "action": "swipe",
                "coordinate1": [int(float(x1)*img_width),int(float(y1)*img_height)],
                "coordinate2": [int(float(x2)*img_width),int(float(y2)*img_height)]
            }
            return json_answer
            # return f"scroll(start_box='<|box_start|>({x1},{y1})<|box_end|>', end_box='<|box_start|>({x2},{y2})<|box_end|>')"
    
    return "Action not recognized"


def is_gt_data(data_list):
    return all(
        data['is_correct'] == 'Y' or (data['is_correct'] == 'N' and data['human_action'])
        for data in data_list
    )

def is_trajectory_data(data_list):
    if data_list[-1]['is_correct'] == 'Y' and data_list[-1]['action'] == 'Stop':
        return True
    if data_list[-1]['is_correct'] == 'N' and data_list[-1]['human_action'] == 'Stop':
        return True
    return False

def extract_between_answer(text):
    start = text.find("<answer>") + len("<answer>")
    end = text.find("</answer>")
    if start != -1 and end != -1:
        return text[start:end].strip()
    return ""

def extract_between_think(text):
    start = text.find("<think>") + len("<think>")
    end = text.find("</think>")
    if start != -1 and end != -1:
        return text[start:end].strip()
    return ""


def extract_between_tool_call(text):
    start = text.find("<tool_call>") + len("<tool_call>")
    end = text.find("</tool_call>")
    if start != -1 and end != -1:
        return text[start:end].strip()
    return ""

def is_chinese(text):
    # 匹配中文字符的正则表达式（包括简体、繁体）
    return bool(re.search(r'[\u4e00-\u9fff]', text))

def is_english(text):
    # 使用正则表达式匹配英文文本
    # 允许英文字母、数字、标点符号和空白字符
    pattern = r'^[a-zA-Z0-9\s.,!?\'\"()\-:;]*$'
    return bool(re.match(pattern, text))

def split_tool_call(text: str) -> tuple[str, str, str]:
    """
    将文本拆分成三段：
    before  : <tool_call> 之前的字符串
    middle  : <tool_call> 与 </tool_call> 之间的字符串（去掉首尾空白）
    after   : </tool_call> 之后的字符串

    如果标记不存在，则 middle 为空串，before 为原始文本，after 为空串。
    """
    open_tag, close_tag = "<tool_call>", "</tool_call>"

    start = text.find(open_tag)
    end   = text.find(close_tag, start + len(open_tag))  # 从 open_tag 之后再找，避免嵌套误匹配

    # 没找到成对标记，直接返回
    if start == -1 or end == -1:
        return text, "", ""

    before = text[:start]
    middle = text[start + len(open_tag) : end].strip()
    after  = text[end + len(close_tag) :]

    return before, middle, after


# Inquiry Data

In [3]:
instruction2app = {}

inquiry_data_1 = read_jsonl("/home/aiqihang.aqh/Appagent/data/【iTAG】交互原因交互内容标注_0530_v2_UTF__20250628102014.jsonl")
inquiry_data_2 = read_jsonl("/home/aiqihang.aqh/Appagent/data/interactive_added.jsonl")

for data in instruction_data:
    instruction2app[data['instruction']] = data['apps']

for idx,data in enumerate(inquiry_data_1):
    try:
        inquiry_data_1[idx]['apps'] = instruction2app[data['instruction']]
    except:
        print(f"{idx}: {data['instruction']}")

inquiry_data_1[888]['apps'] = ['xhs']

888: 打开小红书，发一条笔记，内容是“鹅毛大雪”，并用小红书自带功能生成配图。


In [11]:
ast.literal_eval(inquiry_data_1[0]['介入原因'])

['风险场景']

In [5]:
category = []

category_unified = {
    "风险场景": "risk",
    "隐私安全": "privacy",
    "意图确认": "intension",
    "其他": "uncertainty"
}


for idx, data in enumerate(inquiry_data_1):
    raw = data['介入原因']
    try:
        cats = ast.literal_eval(raw)
        if isinstance(cats, list):
            if len(cats) > 1:
                category += ['combination']
                inquiry_data_1[idx]['category'] = 'combination'
            else:
                inquiry_data_1[idx]['category'] = category_unified[cats[0]]
                category += cats
        else:
            print(f"Warning: Entry {idx} is not a list: {raw!r}")
    except Exception as e:
        print(f"Error parsing entry {idx}: {raw!r}\nException: {e}")

for data in inquiry_data_2:
    category.append(data['category'])


category_cnt = {}

for idx,data in enumerate(category):
    if data in category_unified:
        category[idx] = category_unified[data]

    if category[idx] in category_cnt:
        category_cnt[category[idx]] += 1
    else:
        category_cnt[category[idx]] = 1

category_cnt

{'risk': 52,
 'privacy': 145,
 'intension': 571,
 'uncertainty': 127,
 'combination': 80}

In [30]:
all_inquiry_data = inquiry_data_1+inquiry_data_2

app_unified = {
    "weixin": "wechat",
    "微信": "wechat",
    "QQ": "qq",
    "淘宝": "taobao",
    "微博": "weibo",
    "百度网盘": "baidu netdisk",
    "baidu_netdisk": "baidu netdisk",
    "银行": "bank",
    "爱奇艺": "iqiyi",
    "美团": "meituan",
    "抖音": "tiktok",
    "douyin": "tiktok",
    "优酷": "youku",
    "京东": "jd",
    "腾讯视频": "wetv",
    "网易云音乐": "netease music",
    "netease cloudmusic": "netease music",
    "支付宝": "alipay",
    "未知": "unknown",
    "xiaohongshu": "rednote",
    "xhs": "rednote"
}

for idx,data in enumerate(all_inquiry_data):
    new_apps = []

    for app in data['apps']:
        if app in app_unified:
            new_apps.append(app_unified[app])
        else:
            new_apps.append(app)

    all_inquiry_data[idx]['apps'] = new_apps

all_apps = []
all_instructions = []
instruction2category = {}

en_data = []
zh_data = []

for data in all_inquiry_data:
    all_apps += data['apps']
    all_instructions.append(data['instruction'].strip())
    instruction2category[data['instruction'].strip()] = data['category']

    if is_chinese(data['instruction'].strip()):
        zh_data.append(data)
    else:
        en_data.append(data)


all_apps = list(set(all_apps))
all_instructions = list(set(all_instructions))

for instruction in all_instructions:
    if is_chinese(instruction):
        zh_data.append(instruction)
    else:
        en_data.append(instruction)



print(f"all apps: {len(all_apps)}")
print(f"all instructions {len(all_instructions)}")
print(f"en data: {len(en_data)}, zh data: {len(zh_data)}")

all apps: 37
all instructions 173
en data: 368, zh data: 780


In [32]:
category_app = {
    "risk": [],
    "privacy": [],
    "intension": [],
    "combination": [],
    "uncertainty": []
}


category_instruction = {
    "risk": 0,
    "privacy": 0,
    "intension": 0,
    "combination": 0,
    "uncertainty": 0
}


en_inquiry_data = []
zh_inquiry_data = []


# for data in all_inquiry_data:
#     category_instruction[data['category']].append(data['instruction'])
#     if is_chinese(data['instruction']):
#         category_instruction[data['category']]["zh"].append(data['instruction'])
#         zh_inquiry_data.append(data)
#     else:
#         category_instruction[data['category']]["en"].append(data['instruction'])
#         en_inquiry_data.append(data)
#     for app in data['apps']:
#         category_app[data['category']].append(app)
for data in all_inquiry_data:
    instruction = data['instruction'].strip()
    category = data['category']
    apps = data['apps']
    for app in apps:
        category_app[category].append(app)


for instruction in instruction2category:
    category_instruction[instruction2category[instruction]] += 1



category_app_top3 = {
    "risk": [],
    "privacy": [],
    "intension": [],
    "combination": [],
    "uncertainty": []
}

from collections import Counter

for key, value in category_app.items():
    # 统计每个 app 出现的次数
    counter = Counter(value)
    
    # 获取出现次数最多的前 3 个 app
    top_3 = counter.most_common(4)
    
    print(f"Category: {key}")
    print("Top 3 apps:")
    for app, count in top_3:
        print(f"  {app}: {count} times")
    print("-" * 40)


Category: risk
Top 3 apps:
  wechat: 36 times
  bilibili: 5 times
  wetv: 4 times
  jd: 3 times
----------------------------------------
Category: privacy
Top 3 apps:
  wechat: 49 times
  alipay: 22 times
  baidu netdisk: 17 times
  weibo: 11 times
----------------------------------------
Category: intension
Top 3 apps:
  unknown: 434 times
  tiktok: 79 times
  rednote: 72 times
  taobao: 71 times
----------------------------------------
Category: combination
Top 3 apps:
  unknown: 18 times
  wetv: 10 times
  iqiyi: 8 times
  youku: 8 times
----------------------------------------
Category: uncertainty
Top 3 apps:
  rednote: 25 times
  taobao: 15 times
  weibo: 15 times
  phone: 12 times
----------------------------------------


In [34]:
category_instruction

{'risk': 12,
 'privacy': 33,
 'intension': 81,
 'combination': 22,
 'uncertainty': 25}